# MC-FPET pilot on Kaggle: extra seeds for cifar and dtd
Runs only the configs listed in `RUNS` (any of adaptformer, fpet-r2, mcpet-r2, fpet-r4, mcpet-r4) for each seed in `SEEDS` and dataset in `DATASETS`.
Finished runs are skipped, so the notebook can be re-run after a timeout.

Settings: GPU T4 x2, Internet on, input dataset `fpet-assets` (must contain cifar and dtd).
To resume after a timeout, also attach the previous version's output as an input: cell 3 restores its finished runs.


In [ ]:
# 0. what to run
SEEDS = [2]
DATASETS = ["cifar"]
RUNS = ["fpet-r4", "mcpet-r4"]

# evaluate only 4 times (epochs 24/49/74/99) instead of every 10 epochs
EVAL_EVERY = 25

In [ ]:
# 1. code
import os
if not os.path.isdir("/kaggle/working/kims_fpet"):
    !git clone -b merge-consistency https://github.com/kelvenniky/kims_fpet.git /kaggle/working/kims_fpet
%cd /kaggle/working/kims_fpet
!pip install -q timm==0.4.12

In [ ]:
# 2. weights + data
import os, glob
# works for any account and any nesting (Kaggle nests uploads as name/name)
A = sorted(glob.glob("/kaggle/input/**/fpet-assets", recursive=True), key=len)[0]
print("assets:", A)

# ViT-B_16.npz may itself be a folder holding the real file, so pick the actual file
if not os.path.isfile("ViT-B_16.npz"):
    if os.path.lexists("ViT-B_16.npz"):
        os.remove("ViT-B_16.npz")
    files = [p for p in glob.glob(f"{A}/**/ViT-B_16.npz", recursive=True) if os.path.isfile(p)]
    if files:
        os.symlink(files[0], "ViT-B_16.npz")
    else:
        !wget -q https://storage.googleapis.com/vit_models/imagenet21k/ViT-B_16.npz
!ls -laL ViT-B_16.npz

# uploaded folders can be nested (dtd/dtd), so link the real ones into a clean root
os.makedirs("/kaggle/working/vtab", exist_ok=True)
for name in DATASETS:
    hits = glob.glob(f"{A}/**/{name}/train800val200.txt", recursive=True)
    assert hits, f"{name} not found in {A}: add it to the fpet-assets dataset"
    link = f"/kaggle/working/vtab/{name}"
    if not os.path.exists(link):
        os.symlink(os.path.dirname(hits[0]), link)
    print(name, "->", os.path.dirname(hits[0]))
os.environ["VTAB_ROOT"] = "/kaggle/working/vtab"


In [ ]:
# 3. (no restore: every run in RUNS is trained from scratch)


In [ ]:
# 4. run fpet-r4 and mcpet-r4 (no check for existing results); results are zipped after every run
%cd /kaggle/working/kims_fpet
FLAGS = {"fpet-r4": "--r_layer 4", "mcpet-r4": "--r_layer 4 --consistency"}
for seed in SEEDS:
    for dataset in DATASETS:
        for run in RUNS:
            d = f"result/pilot/{run}/seed{seed}"
            os.makedirs(f"{d}/{dataset}", exist_ok=True)
            !CUDA_VISIBLE_DEVICES=0 python main.py --dataset {dataset} --method adaptformer --dim 8 --bit 32 --load_config --seed {seed} --model_path {d} {FLAGS[run]} --eval_every {EVAL_EVERY} 2>&1 | tee {d}/{dataset}/stdout.log
            !zip -qr /kaggle/working/pilot_results.zip result/pilot
            !cat {d}/{dataset}/summary.json
!find result -name summary.json | sort
